# Open Bandit A/B testing: run the pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RollinsE/open-bandit-ab-testing/blob/main/notebooks/00_run_pipeline_colab.ipynb)

This notebook is the supported Colab entrypoint. It runs the six stages:

1. Data loading and validation
2. A/B test: Thompson Sampling vs random
3. Reward model marathon: click models compete; the simplest model within a margin of the best wins
4. Off-policy evaluation benchmark with each campaign's winning click model, then fair estimator selection
5. A/B vs off-policy comparison
6. Reporting

Every run writes to `outputs/builds/<run_id>/` with its own log file, artifacts and report.

In [ ]:
# Setup: clone the repository (Colab) or use the local checkout, then install.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/RollinsE/open-bandit-ab-testing.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_DIR = Path("/content/open-bandit-ab-testing")
    if not PROJECT_DIR.exists():
        !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
    elif (PROJECT_DIR / ".git").exists():
        !git -C {PROJECT_DIR} pull --ff-only -q  # pick up any updates in a session that is still running
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
%pip install -q -r requirements/colab.txt
print("Project folder:", PROJECT_DIR)

## 1. Choose your data and where outputs go

Leave everything as it is to try the pipeline on the bundled sample (about five minutes).

For the **full** Open Bandit Dataset (about 27M rows), download `open_bandit_dataset.zip` from https://research.zozo.com/data.html, put the zip on Google Drive, and set `USE_DRIVE = True` and `DATA_ZIP` to its path. The zip is unpacked to the Colab disk once per session, and outputs go to Drive so nothing is lost if Colab disconnects. A full run takes about 1.5 hours; the log shows the time left as it goes.

In [ ]:
USE_DRIVE = False   # True: keep outputs on Google Drive (recommended for the full dataset)
DATA_ZIP = None     # e.g. "/content/drive/MyDrive/open-bandit/open_bandit_dataset.zip"; None uses the bundled sample
CAMPAIGNS = "all men women"

OUTPUT_DIR = "outputs"
DATA_DIR = None
if IN_COLAB:
    OUTPUT_DIR = "/content/outputs"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/open-bandit/outputs"
if DATA_ZIP:
    unpacked = Path("/content/obd_full" if IN_COLAB else "data/obd_full")
    found = list(unpacked.glob("**/random/all/all.csv")) if unpacked.exists() else []
    if not found:
        print("Unpacking the dataset (a few minutes)...")
        !mkdir -p {unpacked} && unzip -q -o "{DATA_ZIP}" -d {unpacked}
        found = list(unpacked.glob("**/random/all/all.csv"))
    if not found:
        raise FileNotFoundError(f"No random/all/all.csv inside {DATA_ZIP}")
    DATA_DIR = str(found[0].parents[2])
print("Data:", DATA_DIR or "bundled sample")
print("Outputs:", OUTPUT_DIR)

## 2. Run the pipeline

**If Colab disconnects, re-run the cells above and then this one again.** It picks up the unfinished build where it stopped, with the settings it started with, and skips everything already done; when there is nothing unfinished it starts a new build. If it says another session is working on the build, wait two minutes for the old session's lock to expire.

The progress lines in the log show how far along each stage is and roughly how long is left.

In [ ]:
data_arg = f"--data-dir {DATA_DIR}" if DATA_DIR else ""
!python cli.py --mode colab --output-dir {OUTPUT_DIR} --campaigns {CAMPAIGNS} {data_arg} --continue-build auto --promote-release

## Results from the latest build

In [ ]:
import pandas as pd
from IPython.display import HTML, Markdown, display

# The promoted release is the most recent completed run.
build = Path(OUTPUT_DIR) / "releases" / "latest"
if not build.exists():
    build = sorted(Path(OUTPUT_DIR, "builds").iterdir())[-1]
print("Build:", build)

ab = pd.read_csv(build / "artifacts" / "ab_test_results.csv")
display(ab[["campaign", "control_ctr", "treatment_ctr", "rel_lift", "rel_lift_lower",
            "rel_lift_upper", "p_value", "achieved_power", "decision"]].round(4))

# Interactive Plotly figures: hover for values, click the legend to hide a series
# Model marathon leaderboard, best first
board = pd.read_csv(build / "artifacts" / "model_performance.csv")
display(board[["campaign", "Model", "Type", "Log loss", "ROC AUC", "Skill vs baseline", "winner"]].round(5))

# Fair estimator selection: chosen on one slice, scored on another
selection = pd.read_csv(build / "artifacts" / "estimator_selection.csv")
display(selection[["split", "held_out", "selected_on", "selected_estimator", "test_error",
                   "oracle_estimator", "oracle_error", "median_error"]].round(4))

for fig in ("ab_ctr_by_campaign", "ab_lift", "ab_sample_size", "model_leaderboard", "ope_relative_error",
            "estimator_selection", "ab_vs_ope_lift"):
    display(HTML((build / "reports" / "figures" / f"{fig}.html").read_text()))

In [ ]:
# The full written report (figures are linked; open reports/report.html for the version with charts embedded)
display(Markdown((build / "reports" / "report.md").read_text()))

In [ ]:
# Last lines of this run's log file
log_file = next((build / "logs").glob("*.log"))
print("".join(log_file.read_text().splitlines(True)[-25:]))

## Plan your own A/B test

The A/B functions work on any two-arm click test. Enter your own counts to analyse a result, or a baseline CTR to size a new test.

In [ ]:
from dataclasses import asdict
from src.ab_testing import analyse_ab_test, required_sample_size

result = analyse_ab_test("my test", x_c=480, n_c=100_000, x_t=560, n_t=100_000, random_state=0)
display(pd.Series(asdict(result)).to_frame("value"))

baseline_ctr = 0.004
for lift in (0.05, 0.10, 0.20):
    n = required_sample_size(baseline_ctr, baseline_ctr * (1 + lift))
    print(f"To detect a {lift:.0%} lift from {baseline_ctr:.2%} CTR: {n:,.0f} impressions per arm")